# Phase 2 - Part B: Generative AI Integration

## 1. Generative AI Model Setup

We use **LLaMA 3.3 70B** (Meta's open-source LLM) accessed through the **Groq API**.

**Why Groq + LLaMA?**
- Free API access (no credit card required)
- Open-source model aligns with academic transparency
- Fast inference via Groq's specialized hardware
- Listed as an option in the project handbook


In [5]:
# Import required libraries
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env file
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Initialize the Groq client
client = Groq(api_key=api_key)

# Model configuration
MODEL_NAME = "llama-3.3-70b-versatile"

# Quick connection test
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Reply with: connection OK"}],
    max_tokens=10
)

print("Model:", MODEL_NAME)
print("Response:", response.choices[0].message.content)

Model: llama-3.3-70b-versatile
Response: connection OK


## 2. Prompt Template Design

### Template 1: Simple Explanation

**Goal:** Explain the prediction to the patient in plain, friendly language.

**Target user:** Patients with no medical background.

**Design choice:** Short response (3-5 sentences), no medical jargon, reassuring tone.

In [12]:
# Template 1: Simple Explanation
# Goal: explain results in plain language for non-medical users

template_1_system = """You are a friendly health assistant. 
Explain medical results in simple, everyday language. 
Avoid technical terms. Keep responses short (3-5 sentences).
Be reassuring but honest."""

template_1_user = """A patient received their liver health screening result.

Prediction: {prediction}
Key values:
{features}

Explain this result in plain language, as if talking to a friend with no medical background.
Focus on the overall meaning, not the numbers."""

In [13]:
def simple_explanation(prediction, features):
    """Generate a simple explanation using Template 1."""
    
    user_message = template_1_user.format(
        prediction=prediction,
        features=features
    )
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": template_1_system},
            {"role": "user", "content": user_message}
        ],
        temperature=0.7,
        max_tokens=300
    )
    
    return response.choices[0].message.content

In [14]:
import pandas as pd

# Load raw dataset (easier for the AI to interpret)
data = pd.read_csv("Raw_Dataset/indian_liver_patient.csv")

# 3 diverse test cases
test_indices = [0, 315, 440] 

for i, idx in enumerate(test_indices, start=1):
    patient = data.iloc[idx]
    
    # Convert prediction: 1 = Liver Disease, 2 = No Liver Disease
    prediction = "Liver Disease Detected" if patient["Dataset"] == 1 else "No Liver Disease"
    
    # Format features (drop the label so the AI doesn't see the answer)
    features = "\n".join([
        f"- {col}: {val}"
        for col, val in patient.drop("Dataset").items()
    ])
    
    # Generate explanation
    print(f"Patient {i}: (row {idx}), Prediction: {prediction}")
    
    result = simple_explanation(prediction, features)
    print(result)

Patient 1: (row 0), Prediction: Liver Disease Detected
Hey, I know getting test results can be nerve-wracking. Unfortunately, your liver health screening shows that there might be some issues with your liver. This doesn't necessarily mean it's serious, but it's something we should look into further. We'll likely need to do some more tests to understand what's going on and figure out the best way to take care of your liver.
Patient 2: (row 315), Prediction: No Liver Disease
Hey, great news - your liver health screening results look good. The test shows that you don't have any signs of liver disease, which is a big relief. Your liver is working properly and everything seems to be in order. You can feel good about taking care of your body, and just keep up with your healthy habits. Overall, it's a positive result, so you can breathe easy.
Patient 3: (row 440), Prediction: Liver Disease Detected
Hey, I know this might be a bit concerning, but let's break it down. Your liver health screenin